# Validacion de nombre geografico por imagen

Notebook minimo para validar que el nombre esperado de una imagen se construye solo con la fecha del archivo y el sector obtenido por cruce geografico contra el feature class de indice de vuelos. El nombre del sector se preserva como viene en el feature class, reemplazando espacios por `_`.

In [6]:
from datetime import datetime
from pathlib import Path
import importlib

import pandas as pd

import core.mosaic_image_audit as mosaic_audit
mosaic_audit = importlib.reload(mosaic_audit)
from core.mosaic_image_audit import *

# PARAMETROS
# Puedes dejar una sola imagen o agregar mas rutas a la lista.
PATH_IMAGENES = [
    '\\\\amssclgis10.ams.gmams.cl\\CL_MLP_PAO\\Vuelos_Drone_Sin_Procesar\\INPUT\\20260519_Geosupport\\SOLO_TIF_19May26\\GEOSP-TRN-002511_GS_Ortofoto Estación Cabeceras (EC)_01_05_26.tif',
]

PATH_FC_INDICE_VUELOS_IMGS = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO"
SECTOR_FIELD_INDICE_VUELOS = "Sector"
QUERY_INDICE_VUELOS = "Sensor <> 'DJI MATRICE 350 RTK'"  # Ejemplo: "Tipo = 'Ortofoto'"

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = Path.cwd() / "outputs" / "validacion_nombre_geografico"
output_dir.mkdir(parents=True, exist_ok=True)

print("Modulo auditoria:", mosaic_audit.__file__)
print("Version logica:", AUDIT_LOGIC_VERSION)
print("Feature class indice vuelos:", PATH_FC_INDICE_VUELOS_IMGS)
print("Campo sector:", SECTOR_FIELD_INDICE_VUELOS)
print("Query indice vuelos:", QUERY_INDICE_VUELOS)

Modulo auditoria: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\core\mosaic_image_audit.py
Version logica: 2026-06-15-spatial-sector-preserve-token
Feature class indice vuelos: \\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO
Campo sector: Sector
Query indice vuelos: Sensor <> 'DJI MATRICE 350 RTK'


## 1. Preparar imagenes a validar

Solo se usa el nombre del archivo para extraer la fecha. El sector no se infiere desde el nombre.

In [7]:
paths = [path for path in PATH_IMAGENES if str(path).strip()]
if not paths:
    raise ValueError("Configura al menos una ruta en PATH_IMAGENES")

image_rows = []
date_rows = []

for image_path_text in paths:
    image_path = Path(image_path_text)
    date_info = extract_date_token_from_filename(image_path.name)
    image_rows.append(
        {
            "file_name": image_path.name,
            "stem": image_path.stem,
            "extension": image_path.suffix.lower(),
            "path": str(image_path),
            "relative_path": image_path.name,
        }
    )
    date_rows.append(
        {
            "file_name": image_path.name,
            "date_found": date_info is not None,
            "date_token": date_info.get("date_token") if date_info else None,
            "date_match_text": date_info.get("matched_text") if date_info else None,
            "date_warning": date_info.get("date_warning") if date_info else None,
        }
    )

imagenes_df = pd.DataFrame(image_rows)
fechas_df = pd.DataFrame(date_rows)

display(imagenes_df)
display(fechas_df)

,file_name,stem,extension,path,relative_path
0,GEOSP-TRN-002511_GS_Ortofoto Estación Cabecera...,GEOSP-TRN-002511_GS_Ortofoto Estación Cabecera...,.tif,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,GEOSP-TRN-002511_GS_Ortofoto Estación Cabecera...


,file_name,date_found,date_token,date_match_text,date_warning
0,GEOSP-TRN-002511_GS_Ortofoto Estación Cabecera...,True,26_05_01,01_05_26,None


## 2. Calcular sector por cruce geografico

Si la imagen cruza mas de un sector, se selecciona el sector con mayor porcentaje de interseccion.

In [8]:
spatial_matches_df = calculate_spatial_sector_matches(
    imagenes_df,
    PATH_FC_INDICE_VUELOS_IMGS,
    sector_field=SECTOR_FIELD_INDICE_VUELOS,
    where_clause=QUERY_INDICE_VUELOS,
)

display_columns = [
    "file_name",
    "spatial_status",
    "spatial_sector_raw",
    "spatial_sector",
    "spatial_overlap_pct",
    "spatial_overlap_count",
    "spatial_all_matches",
    "spatial_error",
]
display_columns = [column for column in display_columns if column in spatial_matches_df.columns]

display(spatial_matches_df[display_columns])

,file_name,spatial_status,spatial_sector_raw,spatial_sector,spatial_overlap_pct,spatial_overlap_count,spatial_all_matches,spatial_error
0,GEOSP-TRN-002511_GS_Ortofoto Estación Cabecera...,ok,Estacion_Cabecera,Estacion_Cabecera,46.146846,5,Estacion_Cabecera:46.15|Estacion_Cabecera:46.0...,NaN


## 3. Generar nombre esperado

El nombre esperado se genera con `CL_MLP_PAO_IF_Ortho` + fecha del archivo + sector geografico. El sector conserva el texto del feature class y solo cambia espacios por `_`. Si no hay cruce espacial valido, el nombre queda vacio para revision.

In [ ]:
nombre_geografico_df = add_expected_names_with_spatial_sector(
    imagenes_df,
    spatial_matches_df,
)

result_columns = [
    "file_name",
    "path",
    "expected_file_name",
    "expected_name",
    "expected_date_token",
    "expected_sector",
    "sector_source",
    "rename_status",
    "spatial_status",
    "spatial_sector_raw",
    "spatial_sector",
    "spatial_overlap_pct",
    "spatial_overlap_count",
    "spatial_all_matches",
    "date_warning",
]
result_columns = [column for column in result_columns if column in nombre_geografico_df.columns]

display(nombre_geografico_df[result_columns])

## 4. Exportar resultado

Se deja un CSV simple para comparar fuera del servidor.

In [ ]:
output_csv = output_dir / f"nombre_geografico_{run_timestamp}.csv"
nombre_geografico_df[result_columns].to_csv(output_csv, index=False, encoding="utf-8-sig")

print("CSV exportado:", output_csv)